# Exploring the performance of Isolation Forests for Anomaly Detection

### Imports

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from relaiss import constants
import relaiss as rl

from alerce.core import Alerce
al = Alerce()

### Load data

In [100]:
csv_path = "/Users/jennakempster-taylor/re-laiss/reference_20k_with_durations.csv"
df = pd.read_csv(csv_path, low_memory=False)

# Examine shape of df
print("Shape of durations df:", df.shape) 

# RELAISS default features
default_lc_features = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

print("Default LC features (sample):", default_lc_features)
print("Default host features (sample):", default_host_features)

Shape of durations df: (25515, 462)
Default LC features (sample): ['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_max_rolling_variance', 'r_max_rolling_variance', 'g_mean_rolling_variance', 'r_mean_rolling_variance', 'g_rise_local_curvature', 'g_decline_local_curvature', 'r_rise_local_curvature', 'r_decline_local_curvature']
Default host features (sample): ['gKronMagCorrected', 'gKronRad', 'gExtNSigma', 'rKronMagCorrected', 'rKronRad', 'rExtNSigma', 'iKronMagCorrected', 'iKronRad', 'iExtNSigma', 'zKronMagCorrected', 'zKronRad', 'zExtNSigma', 'gminusrKronMag', 'rminusiKronMag', 'iminuszKronMag', 'rmomentXX', 'rmomentXY', 'rmomentYY']


In [101]:
# Missing data summary

added_cols = ['antares_duration', 'duration_days', 'antares_newest_alert', 'antares_oldest_alert']

def summarise_missing(df, cols):
    return (
        df[cols].isna().sum().to_frame('Missing')
        .assign(Total=len(df))
        .assign(Percent=lambda x: x['Missing'] / x['Total'] * 100)
    )

missing_summary = summarise_missing(df, added_cols)
print(missing_summary)

                      Missing  Total    Percent
antares_duration         3756  25515  14.720752
duration_days            3756  25515  14.720752
antares_newest_alert       48  25515   0.188125
antares_oldest_alert       48  25515   0.188125


### Process data and introduce cuts:

In [103]:
# Helper functions for this section:

from astropy.coordinates import SkyCoord
import astropy.units as u
from sklearn.impute import KNNImputer

# Find galactic coords:
def compute_galactic_latitudes(df):
    coords = SkyCoord(df["ra"].values * u.deg,
                      df["dec"].values * u.deg,
                      frame="icrs")
    return coords.galactic.b.deg


# Apply cut on duration above 200 days and coords in galactic plane:
def apply_filters(df, max_duration=200, lat_cut=15.0):
    mask_duration = df["duration_days"] <= max_duration
    mask_lat = np.abs(df["gal_b"]) >= lat_cut
    mask = mask_duration & mask_lat
    return df[mask].copy(), mask


def overlap(cols, df):
    return [c for c in cols if (c in df.columns) and (not c.endswith("_err"))]

# Extract features:
def select_feature_columns(df, lc_feats, host_feats=None):
    lc = overlap(lc_feats, df)
    host = overlap(host_feats, df) if host_feats else []
    return lc + host, lc, host

# KNN imputation to avoid NaNs:
def knn_impute(df, feature_cols, n_neighbors=5):
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
    X = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    imputer = KNNImputer(n_neighbors=n_neighbors)
    X_imp = imputer.fit_transform(X)
    return X_imp, numeric_cols

In [107]:
# Apply cut:

# Add galactic latitude into df:
df["gal_b"] = compute_galactic_latitudes(df)

# Apply duration ansd galactic coords cut:
df_cut, mask = apply_filters(df)
print(f"Filtered dataset: {len(df_cut)} rows (out of {len(df)})")

# Feature selection:
USE_HOST = False
feature_cols, lc_cols, host_cols = select_feature_columns(
    df_cut,
    default_lc_features,
    default_host_features if USE_HOST else None
)

print(f"Found {len(lc_cols)} LC features.")
print(f"Total features used: {len(feature_cols)}")

Filtered dataset: 17254 rows (out of 25515)
Found 25 LC features.
Total features used: 25


In [109]:
# KNN imputation:

X_imp, numeric_feature_cols = knn_impute(df_cut, feature_cols)
print(f"X shape after KNN impute: {X_imp.shape}")

X shape after KNN impute: (17254, 25)


### Model: no host features

In [111]:
# Functions for this section:

# Isolation forest:
def train_isolation_forest(X, n_estimators=300, contamination="auto", random_state=42):
    iso = IsolationForest(
        n_estimators=n_estimators,
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1
    )
    iso.fit(X)
    return iso

# Anomaly scores and rank:
def compute_anomaly_outputs(model, X):
    scores = model.decision_function(X)
    pred = model.predict(X)
    anomaly = (pred == -1).astype(int)
    rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)
    return scores, anomaly, rank


# Function to add to df:
def attach_results(df, scores, anomaly, rank):
    df_out = df.copy()
    df_out["iso_score"] = scores
    df_out["iso_anomaly"] = anomaly
    df_out["iso_rank"] = rank
    return df_out


In [56]:
# Model Application:

iso = train_isolation_forest(X_imp)
scores, anomaly, rank = compute_anomaly_outputs(iso, X_imp)

df_fil = attach_results(df_cut, scores, anomaly, rank)

print("Estimated anomaly rate:", anomaly.mean())

Estimated anomaly rate: 0.04225107221513852


In [112]:
# Top:
preview.sort_values("iso_rank").head(10)


,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
9031,ZTF22abewydn,59822.504873,18.979757,14.723200,15.154900,0.778043,False,18.979757,-0.214968,1,1
1620,ZTF20actzmzp,59180.457222,0.997963,12.346022,12.654753,0.023451,False,0.997963,-0.199994,1,2
13814,ZTF24aahkzvn,60389.519734,1.007234,16.399099,15.646200,0.219078,False,1.007234,-0.195116,1,3
2942,ZTF21aanehlz,59269.465845,6.994768,13.308370,13.583603,0.529780,False,6.994768,-0.191229,1,4
11777,ZTF23aapsuva,60119.405752,118.730058,18.977400,18.754801,0.223114,True,118.730058,-0.184165,1,5
466,ZTF20aapchqy,58898.530648,121.733056,17.626400,17.099001,0.831728,False,121.733056,-0.180580,1,6
8174,ZTF22aapkbkl,59751.427049,135.732234,18.993099,18.510201,0.476248,True,135.732234,-0.172130,1,7
12078,ZTF23aatmqvz,60141.426181,441.989340,20.155300,19.810200,0.543308,True,441.989340,-0.169166,1,8
7516,ZTF22aajjqti,59707.430162,5.976447,14.233615,14.221217,0.063208,False,5.976447,-0.168979,1,9
6673,ZTF21acpfndw,59531.487442,0.101817,15.838621,15.937201,0.057640,False,0.101817,-0.168397,1,10


# Model: Host features:

In [176]:
# Add galactic latitude into df:
df["gal_b"] = compute_galactic_latitudes(df)

# Apply duration ansd galactic coords cut:
df_cut, mask = apply_filters(df)
print(f"Filtered dataset: {len(df_cut)} rows (out of {len(df)})")

USE_HOST = True

feature_cols, lc_cols, host_cols = select_feature_columns(
    df_cut,
    default_lc_features,
    default_host_features if USE_HOST else None
)

print(f"Found {len(lc_cols)} LC features.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features.")

print(f"Total features used: {len(feature_cols)}")


Filtered dataset: 17254 rows (out of 25515)
Found 25 LC features.
Found 11 host features.
Total features used: 36


In [178]:
# KNN:
X_imp, numeric_feature_cols = knn_impute(df_cut, feature_cols)

print(f"X shape: original {len(df_cut)} × {len(numeric_feature_cols)}")
print(f"After KNN imputation: {X_imp.shape}")

X shape: original 17254 × 36
After KNN imputation: (17254, 36)


In [179]:
# Train:

iso = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores, anomaly, rank = compute_anomaly_outputs(iso, X_imp)
print("Estimated anomaly rate:", anomaly.mean())


Estimated anomaly rate: 0.0412657934392025


In [180]:
df_fil = attach_results(df_cut, scores, anomaly, rank)

# Build 'out' fresh from df_fil
out = df_fil[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_fil.columns]

preview = pd.concat(
    [
        df_fil[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

preview.sort_values("iso_rank").head(10)

feature_cols_new = feature_cols
numeric_feature_cols_new = numeric_feature_cols

In [174]:
USE_HOST = True  # use only light curves initially

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df_cut)
host_cols = overlap(default_host_features, df_cut) if USE_HOST else []
feature_cols = lc_cols + host_cols


print(f"Found {len(lc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in filtered data.")
print(f"Total candidate features: {len(feature_cols)}")


# Ensure all is nnumeric:
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_cut[c])]
X = df_cut[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print(f"X shape: {X.shape}  -> after KNN impute: {X_imp.shape}")

iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

# Attach to df_filt (same length as scores/anomaly/rank)
df_fil = df_cut.copy()
df_fil["iso_score"] = np.asarray(scores).ravel()
df_fil["iso_anomaly"] = np.asarray(anomaly).ravel()
df_fil["iso_rank"] = np.asarray(rank).ravel()

# Build 'out' from df_filt to keep lengths consistent
out = df_fil[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

# Optional context columns (from df_filt!)
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_fil.columns]

preview = pd.concat(
    [
        df_fil[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

feature_cols_old = feature_cols
numeric_feature_cols_old = numeric_feature_cols

Found 25 LC features in filtered data.
Found 11 host features in filtered data.
Total candidate features: 36
X shape: (17254, 36)  -> after KNN impute: (17254, 36)
Estimated anomaly rate: 0.0412657934392025


,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
466,ZTF20aapchqy,58898.530648,121.733056,17.626400,17.099001,0.831728,False,121.733056,-0.186391,1,1
6673,ZTF21acpfndw,59531.487442,0.101817,15.838621,15.937201,0.057640,False,0.101817,-0.177737,1,2
9031,ZTF22abewydn,59822.504873,18.979757,14.723200,15.154900,0.778043,False,18.979757,-0.163115,1,3
8454,ZTF22aauurbv,59782.285174,32.911817,15.255600,15.126600,0.978501,True,32.911817,-0.162128,1,4
13814,ZTF24aahkzvn,60389.519734,1.007234,16.399099,15.646200,0.219078,False,1.007234,-0.156391,1,5
9788,ZTF22abqajav,59877.116910,19.000174,15.858900,15.849000,-0.186077,True,19.000174,-0.149632,1,6
899,ZTF20achusux,59130.428553,64.935012,16.699350,16.806446,0.201978,True,64.935012,-0.142656,1,7
11078,ZTF23aahfxpr,60057.302303,22.984803,16.141100,16.027000,0.509367,True,22.984803,-0.142338,1,8
2942,ZTF21aanehlz,59269.465845,6.994768,13.308370,13.583603,0.529780,False,6.994768,-0.142286,1,9
2556,ZTF21aagtqna,59248.514954,110.840567,18.550900,18.390396,0.690066,True,110.840567,-0.139505,1,10


In [184]:
print("Features only in old:", set(feature_cols_old) - set(feature_cols_new))
print("Features only in new:", set(feature_cols_new) - set(feature_cols_old))


Features only in old: set()
Features only in new: set()


In [186]:
print("OLD ORDER:", feature_cols_old[:20])
print("NEW ORDER:", feature_cols_new[:20])


OLD ORDER: ['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_max_rolling_variance', 'r_max_rolling_variance', 'g_mean_rolling_variance']
NEW ORDER: ['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_max_rolling_variance', 'r_max_rolling_variance', 'g_mean_rolling_variance']


In [188]:
print("Old numeric features:", numeric_feature_cols_old)
print("New numeric features:", numeric_feature_cols_new)


Old numeric features: ['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate', 'g_max_rolling_variance', 'r_max_rolling_variance', 'g_mean_rolling_variance', 'r_mean_rolling_variance', 'g_rise_local_curvature', 'g_decline_local_curvature', 'r_rise_local_curvature', 'r_decline_local_curvature', 'gKronRad', 'gExtNSigma', 'rKronRad', 'rExtNSigma', 'iKronRad', 'iExtNSigma', 'zKronRad', 'zExtNSigma', 'rmomentXX', 'rmomentXY', 'rmomentYY']
New numeric features: ['g_peak_time', 'r_peak_time', 'g_rise_time', 'g_decline_time', 'r_rise_time', 'r_decline_time', 'g_duration_above_half_flux', 'r_duration_above_half_flux', 'g_amplitude', 'r_amplitude', 'g_skewness', 'r_skewness', 'g_beyond_2sigma', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_ra

In [192]:
# 1. Compute galactic latitude
df["gal_b"] = compute_galactic_latitudes(df)

# 2. Apply cuts
df_cut, mask = apply_filters(df)

# 3. Select LC + host features
FEATURES_HOST = default_host_features if USE_HOST else None
feature_cols, lc_cols, host_cols = select_feature_columns(df_cut, default_lc_features, FEATURES_HOST)

# 4. Impute *in the order of df_cut*
X_imp, numeric_feature_cols = knn_impute(df_cut, feature_cols)

# 5. Train Isolation Forest
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores, anomaly, rank = compute_anomaly_outputs(iso, X_imp)

# 6. Build df_fil *immediately*
df_fil = attach_results(df_cut, scores, anomaly, rank)

# 7. Build preview columns AFTER df_fil exists
context_cols = [c for c in ["t0", "r_duration_above_half_flux", ...] if c in df_fil.columns]

out = df_fil[["ZTFID","r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]]

preview = pd.concat(
    [
        df_fil[["ZTFID"] + context_cols].reset_index(drop=True),
        out.reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))


,ZTFID,t0,r_duration_above_half_flux,ZTFID,r_duration_above_half_flux,iso_score,iso_anomaly,iso_rank
381,ZTF19aanhdwf,58567.273079,17.060266,ZTF19aanhdwf,17.060266,0.140149,0,1.0
5799,ZTF21abycnvx,59462.351042,21.941319,ZTF21abycnvx,21.941319,0.122933,0,2.0
7856,ZTF22aamcadi,59723.410509,NaN,ZTF22aamcadi,NaN,0.115497,0,3.0
8526,ZTF22aavxrbn,59783.266169,12.001308,ZTF22aavxrbn,12.001308,0.128425,0,6.0
684,ZTF20accbsxa,59110.323796,18.845231,ZTF20accbsxa,18.845231,0.116595,0,7.0
2475,ZTF21aagmuoz,59248.227072,15.017917,ZTF21aagmuoz,15.017917,0.098298,0,9.0
2131,ZTF21aaaovvg,59215.556366,15.981609,ZTF21aaaovvg,15.981609,0.123655,0,10.0
1306,ZTF20acphiap,59163.480579,5.076539,ZTF20acphiap,5.076539,0.117793,0,11.0
4531,ZTF21abdleps,59365.308866,8.000012,ZTF21abdleps,8.000012,0.112865,0,12.0
8576,ZTF22aaxkewq,59783.301620,13.961551,ZTF22aaxkewq,13.961551,0.112279,0,13.0
